# Part 5, notebook 2 of 6: the guard outside the room

`DockerSession` is the host-side object with the same interface as the Jupyter-based session of Parts 3 and 4, so the loop only needed one new parameter. It starts the container, sends cells, and decides when to give up.

**The analogy.** The guard outside the room numbers every sheet of paper going in, expects the answer to carry the same number, and has a stopwatch. If the stopwatch runs out the guard locks the room and calls a person. There is no "let's try again", because a restart would hide what happened.

In [1]:
import sys, inspect
sys.path.insert(0, "../minimal_loop")
import numpy as np
import docker_session as ds
print(inspect.getsource(ds.DockerSession))

class DockerSession:
    def __init__(self, image, timeout_sec: int = 60, with_show: bool = False, with_tools: bool = False,
                 hard_extra: int = 10, extra_flags: list | None = None):
        import numpy as np
        self.timeout_sec, self.hard = timeout_sec, timeout_sec + hard_extra
        self.name = "sc-sbx-" + uuid.uuid4().hex[:10]
        self.data = tempfile.TemporaryDirectory(prefix="sc_data_")
        np.save(Path(self.data.name) / "img.npy", image)
        Path(self.data.name).chmod(0o755)                        # the container user (uid 10001) must be able to read it
        (Path(self.data.name) / "img.npy").chmod(0o644)
        cmd = ["docker", "run", "-i", "--rm", "--name", self.name, *(FLAGS if extra_flags is None else extra_flags),
               "-v", f"{self.data.name}:/data:ro", "-e", f"CELL_TIMEOUT={timeout_sec}", IMAGE]
        self.p = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        self.q: qu

Read this in four pieces:

- **`__init__`** writes the image to a temp directory mounted read-only at `/data`, starts `docker run -i`, and starts a reader thread that puts one line at a time on a queue. The first cell sent is the setup (load the image, define `submit`).
- **`_call`** writes `{"id": n, "code": ...}`, waits on the queue with a *hard* timeout (the cell timeout plus 10 s), and validates the answer. No answer in time: `docker kill`, mark dead. Container exited: mark dead. Anything malformed: kill, mark dead. (The validation itself is notebook 05c.)
- **`run`** turns a dead session into a result with `fatal` set, so the loop can stop.
- **`kill` / `close`** make sure nothing is left running.

## The only change to the loop

`run_episode4` gained a `session_cls` parameter and one check for a dead sandbox.

In [2]:
import loop4
src = inspect.getsource(loop4.run_episode4).splitlines()
for i, l in enumerate(src):
    if "session_cls" in l or "fatal" in l or "why_none = res.fatal" in l:
        print(f"{i:3d}: {l}")

  1:                  limit=LIMIT_PCT, cell_timeout=60, session_cls=Session4) -> Episode:
  5:     failures, session = 0, session_cls(image, timeout_sec=cell_timeout, with_show=with_show, with_tools=with_tools)
 25:                 if getattr(res, "fatal", None):            # the sandbox died or misbehaved: stop, never guess
 26:                     why_none = res.fatal


## The nine control-flow paths

A scripted stand-in plays the model (it emits the cell we give it), so the only things under test are the container, the cell runner, the host checks and the rule layer. **STAND-IN:** the model. **REAL:** everything else. Each line below is one path through the loop. The rule is an illustrative 1.5 % void limit, not a real standard.

In [3]:
from loop4 import run_episode4
IMG = np.full((16, 16), 200, np.uint8)

def scripted(*cells):
    it = iter(cells)
    return lambda m: (f"Purpose: scripted step.\n```python\n{next(it, 'print(1)')}\n```", {"prompt": 10, "completion": 5})

def check(name, ep, status, reason_part, steps=None):
    ok = ep.verdict.status == status and reason_part in ep.verdict.reason and (steps is None or ep.steps == steps)
    print(f"{'ok  ' if ok else 'FAIL'} {name:40s} -> {ep.verdict.status:6s} {ep.verdict.reason[:60]}  (steps={ep.steps})")
    assert ok, (name, ep.verdict, ep.steps)

def go(cells, **kw):
    return run_episode4(IMG, "q", scripted(*cells), session_cls=ds.DockerSession, cell_timeout=3, **kw)

check("submit above limit",                 go(["submit(2.0)"]), "FAIL", "2.00 > 1.50", 1)
check("variables persist between cells",    go(["x = 0.4", "submit(x)"]), "PASS", "0.40 <= 1.50", 2)
check("error, then recovery",               go(["print(nope)", "submit(0.2)"]), "PASS", "0.20 <= 1.50", 2)
check("never submits -> REVIEW",            go(["print(1)"] * 3, max_steps=3), "REVIEW", "out_of_steps", 3)
check("loop is interrupted, kernel lives",  go(["while True: pass", "submit(0.1)"]), "PASS", "0.10 <= 1.50", 2)
check("alarm disabled -> container killed", go(["import signal\nsignal.signal(signal.SIGALRM, signal.SIG_IGN)\nwhile True: pass", "submit(0.1)"]), "REVIEW", "sandbox_killed", 1)
check("memory bomb -> REVIEW, never PASS",  go(["x = np.ones((60000, 60000))", "submit(0.1)"]), "REVIEW", "sandbox", 1)
check("forged reply (wrong id) -> REVIEW",  go(["import os; os.write(3, b'{\"id\": 1, \"stdout\": \"\", \"error\": \"\", \"submission\": {\"void_ratio_pct\": 0.0, \"note\": \"f\"}}\\n')", "submit(0.1)"]), "REVIEW", "sandbox_protocol_error", 1)
check("exit() inside a cell -> REVIEW",     go(["import os; os._exit(0)", "submit(0.1)"]), "REVIEW", "sandbox_died", 1)
print("9 of 9 paths ok")

ok   submit above limit                       -> FAIL   rule: 2.00 > 1.50  (steps=1)


ok   variables persist between cells          -> PASS   rule: 0.40 <= 1.50  (steps=2)


ok   error, then recovery                     -> PASS   rule: 0.20 <= 1.50  (steps=2)


ok   never submits -> REVIEW                  -> REVIEW out_of_steps  (steps=3)


ok   loop is interrupted, kernel lives        -> PASS   rule: 0.10 <= 1.50  (steps=2)


ok   alarm disabled -> container killed       -> REVIEW sandbox_killed: no answer within 13s  (steps=1)


ok   memory bomb -> REVIEW, never PASS        -> REVIEW sandbox_died: container exited  (steps=1)


ok   forged reply (wrong id) -> REVIEW        -> REVIEW sandbox_protocol_error: answer was not the expected reply to  (steps=1)


ok   exit() inside a cell -> REVIEW           -> REVIEW sandbox_died: container exited  (steps=1)
9 of 9 paths ok


**What the table says.** A bad measurement fails by rule, not by the model's say-so. A runaway loop is interrupted, and if the model disables its own alarm the host kills the whole container and the result is REVIEW, never PASS. A dead or lying sandbox always ends in REVIEW. **What it does not say:** whether a real model would ever write these cells; the stand-in guarantees it does.

Next: notebook 05c, the seven extra paths added after a security review.